# VQ-VAE-2 Codebook Analysis

Loads a trained Stage-1 VQ-VAE-2 model and computes the following statistics on both the **fine** and **coarse** codebooks:

| Statistic | Description |
|-----------|-------------|
| **Distance concentration (CV)** | Coefficient of variation of query-to-codebook distances; low CV → sampling window is near-random, high CV → semantically meaningful |
| **Range ratio** | (max − min) / mean distance per query; complementary spread metric |
| **Codebook utilisation** | Fraction of codewords used; perplexity vs theoretical maximum |
| **Dead codes** | Codewords never selected over the sample batch |
| **Inter-centroid NN distance** | Minimum distance between any two codebook vectors; measures over-crowding |
| **Usage histogram** | Per-codeword selection frequency (log scale) |

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Ensure project root is on the path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.vq_vae.autoencoders import VQ_VAE_2Layer

# ---------------------------------------------------------------------------
# Configuration — edit here
# ---------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CKPT_PATH = (
    PROJECT_ROOT
    / "checkpoints/stage1"
    / "ToyCar+ToyConveyor+fan+pump+slider+valve"
    / "stage1_e128h128K4096_all_final.pt"
)

# Optional: path to an extracted DCASE2020-Task2 dataset root.
# If the path does not exist the notebook falls back to synthetic encoder inputs.
DATA_PATH = Path("/mnt/ssd/LaCie/dcase2020-task2-dev-dataset")
MACHINE_TYPE = "fan"       # used only when DATA_PATH is valid
N_SAMPLES   = 512          # encoder inputs for the analysis
BATCH_SIZE  = 32

print(f"Device  : {DEVICE}")
print(f"Ckpt    : {CKPT_PATH}")
print(f"Ckpt OK : {CKPT_PATH.exists()}")

Device  : cuda
Ckpt    : /home/lucash/Documents/NTUST/Research/papers/semantic-communication-networks/audDSR/checkpoints/stage1/ToyCar+ToyConveyor+fan+pump+slider+valve/stage1_e128h128K4096_all_final.pt
Ckpt OK : True


## 2. Load VQ-VAE-2 from checkpoint

In [2]:
def load_vq_vae(ckpt_path: Path, device: torch.device) -> VQ_VAE_2Layer:
    """Instantiate VQ_VAE_2Layer from a Stage-1 checkpoint."""
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    model = VQ_VAE_2Layer(
        hidden_channels=(
            ckpt["hidden_channels_coarse"],
            ckpt["hidden_channels_fine"],
        ),
        num_residual_layers=ckpt["num_residual_layers"],
        num_embeddings=(
            ckpt["num_embeddings_coarse"],
            ckpt["num_embeddings_fine"],
        ),
        embedding_dim=(
            ckpt["embedding_dim_coarse"],
            ckpt["embedding_dim_fine"],
        ),
        commitment_cost=0.25,
        decay=0.99,
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval().to(device)

    meta = {
        k: ckpt[k]
        for k in [
            "num_embeddings_fine", "num_embeddings_coarse",
            "embedding_dim_fine",  "embedding_dim_coarse",
            "hidden_channels_fine", "hidden_channels_coarse",
            "n_mels", "target_T", "global_step",
        ]
    }
    print("Checkpoint meta:")
    for k, v in meta.items():
        print(f"  {k:30s}: {v}")
    return model, meta


vq_vae, meta = load_vq_vae(CKPT_PATH, DEVICE)
N_MELS = meta["n_mels"]
T      = meta["target_T"]
K_FINE   = meta["num_embeddings_fine"]
K_COARSE = meta["num_embeddings_coarse"]
print(f"\nK_fine={K_FINE}  K_coarse={K_COARSE}  n_mels={N_MELS}  T={T}")

Checkpoint meta:
  num_embeddings_fine           : 4096
  num_embeddings_coarse         : 4096
  embedding_dim_fine            : 128
  embedding_dim_coarse          : 128
  hidden_channels_fine          : 128
  hidden_channels_coarse        : 128
  n_mels                        : 128
  target_T                      : 320
  global_step                   : 20000

K_fine=4096  K_coarse=4096  n_mels=128  T=320


## 3. Collect encoder outputs (z_sample)

If `DATA_PATH` points to an extracted DCASE2020-Task2 dataset the cell loads real
mel-spectrograms; otherwise it generates synthetic white-noise spectrograms, which
are sufficient to probe the geometry of the trained codebook.

In [ ]:
from torch.utils.data import DataLoader


def collect_z_samples(
    vq_vae: VQ_VAE_2Layer,
    n_samples: int,
    n_mels: int,
    T: int,
    batch_size: int,
    device: torch.device,
    data_path: Path | None = None,
    machine_type: str = "fan",
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Encode spectrograms and return (z_fine, z_coarse) — pre-quantization continuous
    feature tensors of shape (N, embedding_dim_*).
    """
    z_fines, z_coarses = [], []

    use_real = (
        data_path is not None
        and data_path.exists()
        and (data_path / machine_type / "train").exists()
    )

    if use_real:
        from src.data.dataset import DCASE2020Task2LogMelDataset
        ds = DCASE2020Task2LogMelDataset(
            root=str(data_path),
            machine_types=[machine_type],
            target_T=T,
        )
        loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0)
        spec_iter = (batch[0].to(device) for batch in loader)
        source_label = f"real DCASE2020 '{machine_type}' train split"
    else:
        # Synthetic: uniform noise in the typical log-mel dB range [-80, 0]
        def _synthetic_iter():
            remaining = n_samples
            while remaining > 0:
                bs = min(batch_size, remaining)
                yield (torch.rand(bs, 1, n_mels, T, device=device) * 80.0 - 80.0)
                remaining -= bs
        spec_iter = _synthetic_iter()
        source_label = "synthetic (uniform log-mel noise)"

    print(f"Collecting {n_samples} encoder outputs from: {source_label}")

    collected = 0
    with torch.no_grad():
        for specs in spec_iter:
            if specs.dim() == 3:
                specs = specs.unsqueeze(1)       # ensure (B,1,H,W)
            specs = specs[:, :1]                 # mono channel only
            _, _, _, _, z_fine, z_coarse = vq_vae.encode_with_prequant(specs)
            # Flatten spatial dims: (B, C, H, W) -> (B*H*W, C)
            B, C_f, H_f, W_f = z_fine.shape
            _,  C_c, H_c, W_c = z_coarse.shape
            z_fines.append(
                z_fine.permute(0, 2, 3, 1).reshape(-1, C_f).cpu()
            )
            z_coarses.append(
                z_coarse.permute(0, 2, 3, 1).reshape(-1, C_c).cpu()
            )
            collected += B
            if collected >= n_samples:
                break

    z_fine_all   = torch.cat(z_fines,   dim=0)
    z_coarse_all = torch.cat(z_coarses, dim=0)
    # Truncate to desired sample count (in latent-spatial units)
    z_fine_all   = z_fine_all[:n_samples]
    z_coarse_all = z_coarse_all[:n_samples]
    print(f"z_fine   : {tuple(z_fine_all.shape)}")
    print(f"z_coarse : {tuple(z_coarse_all.shape)}")
    return z_fine_all, z_coarse_all


z_fine, z_coarse = collect_z_samples(
    vq_vae,
    n_samples=N_SAMPLES,
    n_mels=N_MELS,
    T=T,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    data_path=DATA_PATH,
    machine_type=MACHINE_TYPE,
)

## 4. Distance-concentration analysis

**Interpretation guide**

| CV value | Meaning |
|----------|---------|
| < 0.1 | Distances are near-uniform → distance-based sampling is essentially random |
| 0.1 – 0.5 | Moderate spread → nearby codewords are meaningfully closer |
| > 0.5 | High spread → strong distance signal, sampling window is semantically informative |

In [3]:
def analyze_distance_concentration(
    codebook: torch.Tensor,   # (K, C)
    z_sample: torch.Tensor,   # (N, C) sample of encoder outputs
) -> dict:
    """
    Measure how concentrated the distance distribution is.
    High concentration = distance-based sampling is less meaningful.

    Key metric: coefficient of variation (CV) of distances.
    Low CV  = concentrated = sampling window is nearly random.
    High CV = spread       = sampling window is semantically meaningful.
    """
    dists = (
        z_sample.pow(2).sum(1, keepdim=True)
        + codebook.pow(2).sum(1)
        - 2 * z_sample @ codebook.t()
    )  # (N, K)

    mean_d = dists.mean(dim=1)   # (N,)
    std_d  = dists.std(dim=1)    # (N,)
    cv     = (std_d / mean_d.clamp(min=1e-6)).mean()

    range_ratio = (
        (dists.max(dim=1).values - dists.min(dim=1).values)
        / mean_d.clamp(min=1e-6)
    ).mean()

    return {
        "cv":           cv.item(),
        "range_ratio":  range_ratio.item(),
        "mean_dist":    mean_d.mean().item(),
        "std_dist":     std_d.mean().item(),
        # Per-query distance tensors for downstream plots
        "_mean_d_per_query": mean_d,
        "_std_d_per_query":  std_d,
        "_cv_per_query":     std_d / mean_d.clamp(min=1e-6),
    }

## 5. Codebook utilisation and inter-centroid geometry

In [ ]:
def codebook_utilisation(
    codebook: torch.Tensor,   # (K, C)
    z_sample: torch.Tensor,   # (N, C)
) -> dict:
    """
    Compute codebook-usage statistics by assigning each z to its nearest codeword.
    Returns usage counts, perplexity, and dead-code fraction.
    """
    K = codebook.shape[0]
    with torch.no_grad():
        # Chunked nearest-neighbour to avoid OOM for large K
        x2 = z_sample.pow(2).sum(1, keepdim=True)   # (N, 1)
        e2 = codebook.pow(2).sum(1)                  # (K,)
        chunk = 256
        best_d = torch.full((z_sample.shape[0],), float("inf"))
        best_i = torch.zeros(z_sample.shape[0], dtype=torch.long)
        for offset in range(0, K, chunk):
            c = codebook[offset : offset + chunk]
            d = x2 + e2[offset : offset + chunk] - 2 * z_sample @ c.t()
            d_min, d_arg = d.min(dim=1)
            upd = d_min < best_d
            best_d = torch.where(upd, d_min, best_d)
            best_i = torch.where(upd, d_arg + offset, best_i)

    counts = torch.bincount(best_i, minlength=K).float()  # (K,)
    probs  = counts / counts.sum().clamp(min=1.0)
    perplexity  = torch.exp(-(probs * (probs + 1e-10).log()).sum()).item()
    utilisation = (counts > 0).float().mean().item()
    dead_codes  = int((counts == 0).sum().item())

    return {
        "perplexity":        perplexity,
        "perplexity_pct":    100.0 * perplexity / K,
        "utilisation":       utilisation,
        "dead_codes":        dead_codes,
        "dead_pct":          100.0 * dead_codes / K,
        "_counts":           counts,
    }


def inter_centroid_stats(
    codebook: torch.Tensor,   # (K, C)
    sample_k: int = 512,
) -> dict:
    """
    Minimum / mean pairwise L2 distance between codebook vectors.
    Uses a random sub-sample of up to `sample_k` codewords to keep it tractable.
    """
    K = codebook.shape[0]
    idx = torch.randperm(K)[:sample_k]
    cb  = codebook[idx].float()
    # Pairwise squared distances
    c2  = cb.pow(2).sum(1)
    D   = c2.unsqueeze(0) + c2.unsqueeze(1) - 2 * cb @ cb.t()  # (k, k)
    D   = D.clamp(min=0).sqrt()
    # Mask diagonal
    D.fill_diagonal_(float("inf"))
    nn_dist = D.min(dim=1).values      # nearest-neighbour per centroid
    return {
        "nn_dist_min":    nn_dist.min().item(),
        "nn_dist_mean":   nn_dist.mean().item(),
        "nn_dist_median": nn_dist.median().item(),
        "nn_dist_std":    nn_dist.std().item(),
        "_nn_dist":       nn_dist,
    }

## 6. Run all analyses

In [ ]:
cb_fine   = vq_vae._vq_fine._embedding.weight.detach().cpu().float()    # (K_f, C_f)
cb_coarse = vq_vae._vq_coarse._embedding.weight.detach().cpu().float()  # (K_c, C_c)

print(f"Codebook shapes  —  fine: {tuple(cb_fine.shape)}   coarse: {tuple(cb_coarse.shape)}")
print(f"z_sample shapes  —  fine: {tuple(z_fine.shape)}    coarse: {tuple(z_coarse.shape)}")
print()

# --- Distance concentration ---
dc_fine   = analyze_distance_concentration(cb_fine,   z_fine)
dc_coarse = analyze_distance_concentration(cb_coarse, z_coarse)

# --- Codebook utilisation ---
cu_fine   = codebook_utilisation(cb_fine,   z_fine)
cu_coarse = codebook_utilisation(cb_coarse, z_coarse)

# --- Inter-centroid geometry ---
ic_fine   = inter_centroid_stats(cb_fine)
ic_coarse = inter_centroid_stats(cb_coarse)

# --- Pretty-print summary ---
def _print_block(title, dc, cu, ic):
    print(f"{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(f"  Distance concentration")
    print(f"    CV (mean over queries)  : {dc['cv']:.4f}")
    print(f"    Range ratio             : {dc['range_ratio']:.4f}")
    print(f"    Mean distance           : {dc['mean_dist']:.4f}")
    print(f"    Std  distance           : {dc['std_dist']:.4f}")
    print(f"  Codebook utilisation")
    print(f"    Perplexity              : {cu['perplexity']:.1f}  ({cu['perplexity_pct']:.1f}% of K)")
    print(f"    Utilisation             : {100*cu['utilisation']:.1f}%")
    print(f"    Dead codes              : {cu['dead_codes']}  ({cu['dead_pct']:.1f}%)")
    print(f"  Inter-centroid (NN) distances")
    print(f"    Min NN distance         : {ic['nn_dist_min']:.4f}")
    print(f"    Mean NN distance        : {ic['nn_dist_mean']:.4f}")
    print(f"    Median NN distance      : {ic['nn_dist_median']:.4f}")
    print(f"    Std  NN distance        : {ic['nn_dist_std']:.4f}")

_print_block(f"Fine   codebook  (K={K_FINE})",   dc_fine,   cu_fine,   ic_fine)
_print_block(f"Coarse codebook  (K={K_COARSE})", dc_coarse, cu_coarse, ic_coarse)

## 7. Visualisation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("VQ-VAE-2 Codebook Analysis", fontsize=14, fontweight="bold")

palette = {"fine": "#4C72B0", "coarse": "#DD8452"}


# --- Row 0: fine codebook ---
# Plot 0: CV per query
axes[0, 0].hist(
    dc_fine["_cv_per_query"].numpy(), bins=60, color=palette["fine"], edgecolor="white", linewidth=0.3
)
axes[0, 0].axvline(dc_fine["cv"], color="red", linewidth=1.5, linestyle="--", label=f"mean CV={dc_fine['cv']:.3f}")
axes[0, 0].set_title(f"Fine — Distance CV per query  (K={K_FINE})")
axes[0, 0].set_xlabel("CV  (std / mean)")
axes[0, 0].set_ylabel("Count")
axes[0, 0].legend(fontsize=9)

# Plot 1: codebook usage histogram (log scale)
counts_f = cu_fine["_counts"].numpy()
axes[0, 1].bar(np.arange(len(counts_f)), np.sort(counts_f)[::-1], color=palette["fine"], width=1.0)
axes[0, 1].set_yscale("log")
axes[0, 1].set_title(
    f"Fine — Usage  (util={100*cu_fine['utilisation']:.0f}%  "
    f"perp={cu_fine['perplexity']:.0f})"
)
axes[0, 1].set_xlabel("Codeword rank (sorted)")
axes[0, 1].set_ylabel("Selection count (log)")

# Plot 2: NN distance histogram
axes[0, 2].hist(
    ic_fine["_nn_dist"].numpy(), bins=50, color=palette["fine"], edgecolor="white", linewidth=0.3
)
axes[0, 2].axvline(
    ic_fine["nn_dist_mean"], color="red", linewidth=1.5, linestyle="--",
    label=f"mean={ic_fine['nn_dist_mean']:.3f}"
)
axes[0, 2].set_title("Fine — Inter-centroid NN distance")
axes[0, 2].set_xlabel("L2 distance to nearest neighbour")
axes[0, 2].set_ylabel("Count")
axes[0, 2].legend(fontsize=9)


# --- Row 1: coarse codebook ---
axes[1, 0].hist(
    dc_coarse["_cv_per_query"].numpy(), bins=60, color=palette["coarse"], edgecolor="white", linewidth=0.3
)
axes[1, 0].axvline(
    dc_coarse["cv"], color="red", linewidth=1.5, linestyle="--",
    label=f"mean CV={dc_coarse['cv']:.3f}"
)
axes[1, 0].set_title(f"Coarse — Distance CV per query  (K={K_COARSE})")
axes[1, 0].set_xlabel("CV  (std / mean)")
axes[1, 0].set_ylabel("Count")
axes[1, 0].legend(fontsize=9)

counts_c = cu_coarse["_counts"].numpy()
axes[1, 1].bar(np.arange(len(counts_c)), np.sort(counts_c)[::-1], color=palette["coarse"], width=1.0)
axes[1, 1].set_yscale("log")
axes[1, 1].set_title(
    f"Coarse — Usage  (util={100*cu_coarse['utilisation']:.0f}%  "
    f"perp={cu_coarse['perplexity']:.0f})"
)
axes[1, 1].set_xlabel("Codeword rank (sorted)")
axes[1, 1].set_ylabel("Selection count (log)")

axes[1, 2].hist(
    ic_coarse["_nn_dist"].numpy(), bins=50, color=palette["coarse"], edgecolor="white", linewidth=0.3
)
axes[1, 2].axvline(
    ic_coarse["nn_dist_mean"], color="red", linewidth=1.5, linestyle="--",
    label=f"mean={ic_coarse['nn_dist_mean']:.3f}"
)
axes[1, 2].set_title("Coarse — Inter-centroid NN distance")
axes[1, 2].set_xlabel("L2 distance to nearest neighbour")
axes[1, 2].set_ylabel("Count")
axes[1, 2].legend(fontsize=9)

fig.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / "codebook_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to results/codebook_analysis.png")

## 8. Side-by-side comparison bar chart

In [ ]:
labels   = ["CV", "Range\nratio", "Perplexity\n(% of K)", "Utilisation\n(%)", "Dead codes\n(%)"]
vals_f   = [
    dc_fine["cv"],
    dc_fine["range_ratio"],
    cu_fine["perplexity_pct"],
    100 * cu_fine["utilisation"],
    cu_fine["dead_pct"],
]
vals_c   = [
    dc_coarse["cv"],
    dc_coarse["range_ratio"],
    cu_coarse["perplexity_pct"],
    100 * cu_coarse["utilisation"],
    cu_coarse["dead_pct"],
]

x     = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 4))
rects_f = ax.bar(x - width / 2, vals_f, width, label=f"Fine (K={K_FINE})",   color=palette["fine"])
rects_c = ax.bar(x + width / 2, vals_c, width, label=f"Coarse (K={K_COARSE})", color=palette["coarse"])

ax.bar_label(rects_f, fmt="%.2f", padding=3, fontsize=8)
ax.bar_label(rects_c, fmt="%.2f", padding=3, fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_title("Fine vs. Coarse codebook — key metrics", fontsize=12)
ax.legend(fontsize=10)
ax.set_ylabel("Value (mixed units — see labels)")
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / "codebook_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Summary table

In [ ]:
try:
    import pandas as pd
    rows = []
    for name, dc, cu, ic in [
        (f"Fine   (K={K_FINE})",   dc_fine,   cu_fine,   ic_fine),
        (f"Coarse (K={K_COARSE})", dc_coarse, cu_coarse, ic_coarse),
    ]:
        rows.append({
            "Level": name,
            "CV (↑ = more spread)": f"{dc['cv']:.4f}",
            "Range ratio": f"{dc['range_ratio']:.4f}",
            "Mean dist": f"{dc['mean_dist']:.4f}",
            "Perplexity": f"{cu['perplexity']:.1f} ({cu['perplexity_pct']:.1f}%)",
            "Utilisation": f"{100*cu['utilisation']:.1f}%",
            "Dead codes": f"{cu['dead_codes']} ({cu['dead_pct']:.1f}%)",
            "NN dist (mean)": f"{ic['nn_dist_mean']:.4f}",
            "NN dist (min)": f"{ic['nn_dist_min']:.4f}",
        })
    df = pd.DataFrame(rows).set_index("Level")
    display(df)
except ImportError:
    print("Install pandas for a formatted summary table.")
    for name, dc, cu, ic in [
        ("Fine", dc_fine, cu_fine, ic_fine),
        ("Coarse", dc_coarse, cu_coarse, ic_coarse),
    ]:
        print(f"{name}: CV={dc['cv']:.4f}, range_ratio={dc['range_ratio']:.4f}, "
              f"perplexity={cu['perplexity']:.1f}, util={100*cu['utilisation']:.1f}%, "
              f"dead={cu['dead_codes']}, nn_mean={ic['nn_dist_mean']:.4f}")

## Interpretation notes

- **CV < 0.1**: distances are near-uniform across the codebook. Anomaly generation via _distant codeword sampling_ degrades to uniform random selection — the sampling window is not geometrically meaningful.
- **CV ≥ 0.2**: sufficient spread that nearby vs. far codewords are distinguishable, making the `anomaly_sampling="distant"` strategy in `AnomalyGeneration` worthwhile.
- **High dead-code %** indicates the codebook is over-sized or under-trained; reducing `num_embeddings` or increasing training iterations may help.
- **Low inter-centroid NN distance** signals codebook collapse / over-crowding in a subspace, which correlates with low perplexity.